# 04 Baseline Probability of Default Model

This notebook trains the first baseline Probability of Default model for the LendingClub credit risk project using logistic regression.

## 1. Import Libraries

In [ ]:
# Core libraries
from pathlib import Path
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn tools for modeling and evaluation
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    precision_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load Clean Modeling Data

In [ ]:
# Set input and output paths
data_path = "../data/processed/loans_clean_v1.csv"
models_dir = Path("../models")
tables_dir = Path("../reports/tables")
figures_dir = Path("../reports/figures")

# Create output folders if they do not already exist
models_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

# Load the cleaned dataset from Milestone 1
loans = pd.read_csv(data_path, low_memory=False)

print(f"Rows: {loans.shape[0]:,}")
print(f"Columns: {loans.shape[1]:,}")

### Analysis: Modeling Dataset

**Observation**  
The baseline model uses `loans_clean_v1.csv`, the cleaned dataset created in Milestone 1.

**Interpretation**  
Using the same cleaned artifact keeps the modeling workflow reproducible and avoids redefining the target or sample inside the modeling notebook.

**Modeling Implication**  
The model results should be interpreted as version 1 baseline results for this specific cleaned dataset.

**Decision**  
Use this file as the source for the first logistic regression Probability of Default model.

## 3. Define Required Columns

In [ ]:
# Target variable
target_column = "default_flag"

# Numeric features for the first baseline model
numeric_features = [
    "loan_amnt",
    "term",
    "int_rate",
    "annual_inc",
    "dti",
    "delinq_2yrs",
    "revol_util",
    "total_acc",
    "open_acc",
    "emp_length",
]

# Categorical features for the first baseline model
categorical_features = [
    "grade",
    "sub_grade",
    "home_ownership",
    "verification_status",
    "purpose",
    "addr_state",
]

# Combine all columns needed for modeling
required_columns = [target_column] + numeric_features + categorical_features

# Check that every required column exists in the dataset
missing_columns = [column for column in required_columns if column not in loans.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns exist.")

# Explicit leakage check for the baseline Probability of Default model
# These variables are not allowed because they are known after loan origination
# or describe payment, recovery, hardship, or settlement outcomes.
known_leakage_columns = [
    "loan_status",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "total_pymnt",
    "total_rec_prncp",
    "total_rec_int",
    "settlement_status",
    "hardship_flag",
    "debt_settlement_flag",
]

# Keyword check catches other payment or recovery fields that should not enter the PD model.
leakage_keywords = [
    "pymnt",
    "payment",
    "recover",
    "settlement",
    "hardship",
]

feature_columns_for_check = numeric_features + categorical_features
explicit_leakage_features = [
    column for column in feature_columns_for_check if column in known_leakage_columns
]
keyword_leakage_features = [
    column
    for column in feature_columns_for_check
    if any(keyword in column.lower() for keyword in leakage_keywords)
]
leakage_features = sorted(set(explicit_leakage_features + keyword_leakage_features))

if leakage_features:
    raise ValueError(f"Leakage features found in model inputs: {leakage_features}")

print("Leakage check passed: no payment, recovery, hardship, settlement, or loan_status fields are used as model features.")

### Analysis: Feature Set

**Observation**  
The first feature set includes basic loan terms, borrower financial variables, credit history variables, and categorical risk indicators.

**Interpretation**  
This is a beginner-friendly baseline feature set. It avoids advanced transformations and obvious post-outcome leakage variables.

**Modeling Implication**  
The baseline model should be useful for learning the end-to-end workflow, but it should not be treated as the final production model.

**Decision**  
Use these selected numeric and categorical variables for version 1 logistic regression.

### Analysis: Leakage Review

**Observation**  
The baseline model feature list was reviewed against known leakage fields, including `loan_status`, payment fields, recovery fields, hardship fields, and settlement fields.

**Interpretation**  
None of the selected model inputs are payment, recovery, hardship, settlement, or final loan outcome variables. The model uses origination-time or underwriting-time fields such as loan amount, term, interest rate, grade, borrower income, credit attributes, and application details.

**Modeling Implication**  
This reduces the risk that the baseline PD model learns from information that would not be available at the time of lending.

**Decision**  
Keep logistic regression and keep the current feature set. The notebook now includes an explicit leakage check to prevent these variables from being added accidentally later.

## 4. Drop Rows with Missing Target

In [ ]:
# Drop rows where the target is missing
rows_before = len(loans)
loans_model = loans.dropna(subset=[target_column]).copy()
rows_after = len(loans_model)

# Convert the target to integer values: 0 = non-default, 1 = default
loans_model[target_column] = loans_model[target_column].astype(int)

print(f"Rows before dropping missing target: {rows_before:,}")
print(f"Rows after dropping missing target: {rows_after:,}")
print("Target value counts:")
print(loans_model[target_column].value_counts().sort_index())
print(f"Default rate: {loans_model[target_column].mean():.2%}")

### Analysis: Target Check

**Observation**  
The target check confirms the number of non-default and default loans used for modeling.

**Interpretation**  
The default rate is the baseline event rate that the model needs to improve on. It also helps identify whether the target is imbalanced.

**Modeling Implication**  
If defaults are much less common than non-defaults, accuracy alone can be misleading. Ranking and probability metrics are needed.

**Decision**  
Evaluate the model using ROC-AUC, Average Precision, Brier Score, and classification metrics at a 0.5 threshold.

## 5. Define X and y

In [ ]:
# Define features and target
feature_columns = numeric_features + categorical_features

X = loans_model[feature_columns]
y = loans_model[target_column]

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

## 6. Train-Test Split

In [ ]:
# Split the data into training and test sets
# stratify=y keeps the default rate similar in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Training rows: {X_train.shape[0]:,}")
print(f"Test rows: {X_test.shape[0]:,}")
print(f"Training default rate: {y_train.mean():.2%}")
print(f"Test default rate: {y_test.mean():.2%}")

### Analysis: Train-Test Split

**Observation**  
The dataset is split into training and test sets with stratification on `default_flag`.

**Interpretation**  
Stratification keeps the default rate similar across the train and test samples.

**Modeling Implication**  
A stable test set makes model evaluation more reliable, especially when the default class is smaller than the non-default class.

**Decision**  
Train the model on the training set and reserve the test set for performance evaluation.

## 7. Build Preprocessing and Modeling Pipeline

In [ ]:
# Numeric preprocessing: fill missing values with the median, then scale
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Categorical preprocessing: fill missing values with the most common value, then one-hot encode
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# Apply the right preprocessing steps to the right columns
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ]
)

# Combine preprocessing and logistic regression into one pipeline
pd_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

pd_model

### Analysis: Pipeline Design

**Observation**  
The preprocessing steps and logistic regression model are combined into one scikit-learn pipeline.

**Interpretation**  
Numeric and categorical variables need different preparation before they can be used by logistic regression.

**Modeling Implication**  
Using a pipeline helps prevent data leakage from preprocessing because imputers, scalers, and encoders are learned from the training set only.

**Decision**  
Use this pipeline as the first baseline Probability of Default modeling approach.

## 8. Train Logistic Regression Model

In [ ]:
# Train the model on the training data
pd_model.fit(X_train, y_train)

print("Logistic regression model trained successfully.")

## 9. Predict Default Probabilities

In [ ]:
# Predict probabilities for the default class on the test set
y_pred_proba = pd_model.predict_proba(X_test)[:, 1]

# Convert probabilities into class predictions using a simple 0.5 threshold
y_pred_05 = (y_pred_proba >= 0.50).astype(int)

print("First 10 predicted default probabilities:")
print(y_pred_proba[:10])

### Analysis: Probability Output

**Observation**  
The model produces a predicted default probability for each loan in the test set.

**Interpretation**  
For credit risk, probabilities are more useful than just predicted classes because loans can be ranked from lower risk to higher risk.

**Modeling Implication**  
The predicted probabilities can support risk-based decisioning, pricing analysis, cutoff testing, and portfolio monitoring.

**Decision**  
Evaluate both probability quality and classification performance.

## 10. Calculate Model Metrics

In [ ]:
# Calculate model performance metrics
metrics = {
    "roc_auc": roc_auc_score(y_test, y_pred_proba),
    "average_precision_pr_auc": average_precision_score(y_test, y_pred_proba),
    "brier_score": brier_score_loss(y_test, y_pred_proba),
    "accuracy_at_0_5": accuracy_score(y_test, y_pred_05),
    "precision_at_0_5": precision_score(y_test, y_pred_05, zero_division=0),
    "recall_at_0_5": recall_score(y_test, y_pred_05, zero_division=0),
}

# Convert metrics to a table and save it
metrics_df = pd.DataFrame([metrics])
metrics_path = tables_dir / "pd_model_metrics_v1.csv"
metrics_df.to_csv(metrics_path, index=False)

display(metrics_df)
print(f"Saved metrics to: {metrics_path}")

### Analysis: Model Metrics

**Observation**  
The baseline model is evaluated using ranking, probability, and threshold-based metrics.

**Interpretation**  
ROC-AUC measures ranking ability, Average Precision focuses on performance for the default class, Brier Score measures probability accuracy, and the 0.5 threshold metrics show simple classification performance.

**Modeling Implication**  
A PD model should not be judged by accuracy alone. The model must rank risk well and produce useful probability estimates.

**Decision**  
Use these metrics as the first benchmark for future model improvements.

## 11. ROC Curve

In [ ]:
# Calculate ROC curve points
fpr, tpr, roc_thresholds = roc_curve(y_test, y_pred_proba)

# Plot and save the ROC curve
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {metrics['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random model")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Baseline PD Model")
plt.legend()
plt.tight_layout()

roc_curve_path = figures_dir / "pd_model_roc_curve_v1.png"
plt.savefig(roc_curve_path, dpi=150)
plt.show()

print(f"Saved ROC curve to: {roc_curve_path}")

## 12. Precision-Recall Curve

In [ ]:
# Calculate precision-recall curve points
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_pred_proba)

# Plot and save the Precision-Recall curve
plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label=f"Average Precision = {metrics['average_precision_pr_auc']:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - Baseline PD Model")
plt.legend()
plt.tight_layout()

pr_curve_path = figures_dir / "pd_model_precision_recall_curve_v1.png"
plt.savefig(pr_curve_path, dpi=150)
plt.show()

print(f"Saved Precision-Recall curve to: {pr_curve_path}")

## 13. Calibration Curve

In [ ]:
# Calculate calibration curve points
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10, strategy="quantile")

# Plot and save the calibration curve
plt.figure(figsize=(7, 5))
plt.plot(prob_pred, prob_true, marker="o", label="Baseline PD model")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
plt.xlabel("Mean Predicted Default Probability")
plt.ylabel("Observed Default Rate")
plt.title("Calibration Curve - Baseline PD Model")
plt.legend()
plt.tight_layout()

calibration_curve_path = figures_dir / "pd_model_calibration_curve_v1.png"
plt.savefig(calibration_curve_path, dpi=150)
plt.show()

print(f"Saved calibration curve to: {calibration_curve_path}")

### Analysis: Model Curves

**Observation**  
The ROC curve, Precision-Recall curve, and calibration curve show different views of model performance.

**Interpretation**  
The ROC curve shows ranking across thresholds, the Precision-Recall curve focuses on default identification, and the calibration curve compares predicted probabilities with observed default rates.

**Modeling Implication**  
A useful PD model should rank borrowers by risk and provide probabilities that are reasonably aligned with actual default outcomes.

**Decision**  
Use these charts as visual benchmarks for future model versions.

## 14. Default Rate by Predicted Risk Decile

In [ ]:
# Create a test results table with actual outcomes and predicted probabilities
test_results = pd.DataFrame({
    "default_flag": y_test.values,
    "predicted_default_probability": y_pred_proba,
})

# Create risk deciles based on predicted default probability
# Decile 1 is lowest predicted risk and decile 10 is highest predicted risk
test_results["risk_decile"] = pd.qcut(
    test_results["predicted_default_probability"],
    q=10,
    labels=False,
    duplicates="drop",
) + 1

# Summarize predicted and actual default risk by decile
risk_decile_table = (
    test_results.groupby("risk_decile")
    .agg(
        number_of_loans=("default_flag", "size"),
        average_predicted_default_probability=("predicted_default_probability", "mean"),
        actual_default_rate=("default_flag", "mean"),
        number_of_actual_defaults=("default_flag", "sum"),
        minimum_predicted_default_probability=("predicted_default_probability", "min"),
        maximum_predicted_default_probability=("predicted_default_probability", "max"),
    )
    .reset_index()
    .sort_values("risk_decile")
)

# Save the risk decile table
risk_decile_path = tables_dir / "pd_risk_deciles_v1.csv"
risk_decile_table.to_csv(risk_decile_path, index=False)

display(risk_decile_table)
print(f"Saved risk decile table to: {risk_decile_path}")

In [ ]:
# Plot observed default rate by predicted risk decile
plt.figure(figsize=(8, 5))
plt.bar(
    risk_decile_table["risk_decile"].astype(str),
    risk_decile_table["actual_default_rate"],
    color="steelblue",
)
plt.xlabel("Predicted Risk Decile")
plt.ylabel("Observed Default Rate")
plt.title("Default Rate by Predicted Risk Decile")
plt.tight_layout()

risk_decile_chart_path = figures_dir / "default_rate_by_decile_v1.png"
plt.savefig(risk_decile_chart_path, dpi=150)
plt.show()

print(f"Saved risk decile chart to: {risk_decile_chart_path}")

### Analysis: Risk Deciles

**Observation**  
Risk deciles group test-set loans from lowest predicted default risk to highest predicted default risk.

**Interpretation**  
If the model is useful, the actual default rate should generally increase as the decile number increases. Small reversals can happen because of sample noise, but the overall pattern should move upward from low-risk deciles to high-risk deciles.

**Modeling Implication**  
A clear upward pattern means the model is ranking loans by risk in a useful way. A flat or highly irregular pattern would suggest weak risk separation and would need further investigation.

**Decision**  
Review `pd_risk_deciles_v1.csv` and `default_rate_by_decile_v1.png` after running the notebook to confirm whether actual default rates increase across deciles.

## 15. Save the Trained Model

In [ ]:
# Save the trained preprocessing + logistic regression pipeline
model_path = models_dir / "pd_logistic_regression_v1.pkl"
joblib.dump(pd_model, model_path)

print(f"Saved model to: {model_path}")

### Milestone 2 Decision

**Observation**  
A complete baseline logistic regression PD model has been trained, evaluated, and saved.

**Interpretation**  
This model establishes the first benchmark for the project using a transparent and beginner-friendly method.

**Modeling Implication**  
Future models should be compared against this baseline before adding complexity.

**Decision**  
Use this logistic regression model as the version 1 baseline for Milestone 2.

## Model Inference and Business Interpretation

This final section explains what the baseline Probability of Default model means from a credit risk and business perspective. It uses the trained logistic regression model, saved metrics, calibration results, and risk decile table from Milestone 2.

### 1. Model Purpose

This is the first baseline Probability of Default model for the LendingClub credit risk project. It is not the final model.

The purpose of this model is to estimate borrower-level probability of default and create a clear benchmark for future model versions. A baseline logistic regression model is useful because it is transparent, easy to explain, and produces probabilities that can be compared with actual default rates.

From a credit risk perspective, the most important output is not simply whether a loan is classified as default or non-default. The more useful output is the predicted default probability, because it allows loans to be ranked from lower risk to higher risk.

### 2. Performance Interpretation

In [ ]:
# Load and display the saved model metrics
metrics_summary = pd.read_csv(tables_dir / "pd_model_metrics_v1.csv")

# Show metrics in a readable vertical format
metrics_summary_display = metrics_summary.T.reset_index()
metrics_summary_display.columns = ["metric", "value"]
metrics_summary_display["value"] = metrics_summary_display["value"].round(4)

metrics_summary_display

**ROC-AUC**  
The baseline model has a ROC-AUC of **0.7094**. This means the model has reasonable ability to rank loans by risk: in general, loans that default receive higher predicted risk scores than loans that do not default. This is a useful first result for a baseline model, although it leaves room for improvement.

**PR-AUC / Average Precision**  
The Average Precision score is **0.3701**. This metric matters because defaults are less frequent than non-defaults. PR-AUC focuses more directly on how well the model identifies the default class, so it is especially useful for credit risk problems with class imbalance.

**Brier Score**  
The Brier Score is **0.1453**. This measures probability accuracy, where lower is better. It tells us how close the predicted default probabilities are to the actual default outcomes. This is important because PD models are used as probability models, not only as classifiers.

**Accuracy, Precision, and Recall at 0.5 Threshold**  
At a 0.5 threshold, the model has accuracy of **0.8019**, precision of **0.5435**, and recall of **0.0481**. The low recall shows that a 0.5 cutoff identifies only a small share of actual defaults. This does not mean the model is useless; it means the 0.5 threshold is not necessarily appropriate for credit risk decisions. In credit risk, predicted probabilities, ranking quality, calibration, and risk deciles are usually more informative than a single default/non-default cutoff.

### 3. Calibration Interpretation

In [ ]:
# Create a calibration summary table from the existing calibration curve data
calibration_summary = pd.DataFrame({
    "calibration_bin": range(1, len(prob_true) + 1),
    "average_predicted_pd": prob_pred,
    "actual_default_rate": prob_true,
})

# Positive gap means the model overpredicts default risk in that bin
# Negative gap means the model underpredicts default risk in that bin
calibration_summary["prediction_gap"] = (
    calibration_summary["average_predicted_pd"]
    - calibration_summary["actual_default_rate"]
)

calibration_summary

The calibration curve compares average predicted PD with actual default rates across probability bands.

In the saved Milestone 2 results, calibration is fairly close overall. In the lowest-risk bin, the model predicts about **4.23%** PD and the actual default rate is about **4.27%**, so it slightly **underpredicts** risk. In the second low-risk bin, it also slightly underpredicts risk.

In the highest-risk bin, the model predicts about **44.46%** PD and the actual default rate is about **44.91%**, so it also slightly **underpredicts** risk among the riskiest loans. Several middle bins show very small overprediction or underprediction.

This matters because Expected Loss later depends on PD. If PD is systematically too low, expected losses will be understated. If PD is systematically too high, expected losses will be overstated. For this first baseline, calibration looks directionally reasonable, but it should still be reviewed and improved before the model is used for serious portfolio decisions.

### 4. Risk Decile Interpretation

In [ ]:
# Load the saved risk decile table
risk_decile_summary = pd.read_csv(tables_dir / "pd_risk_deciles_v1.csv")

# Create a short version for business review
risk_decile_short = risk_decile_summary[[
    "risk_decile",
    "number_of_loans",
    "average_predicted_default_probability",
    "actual_default_rate",
    "number_of_actual_defaults",
]].copy()

risk_decile_short = risk_decile_short.rename(columns={
    "risk_decile": "decile",
    "average_predicted_default_probability": "average_predicted_pd",
    "number_of_actual_defaults": "number_of_defaults",
})

risk_decile_short

In [ ]:
# Calculate simple decile comparison statistics
lowest_decile_default_rate = risk_decile_short.loc[
    risk_decile_short["decile"] == risk_decile_short["decile"].min(),
    "actual_default_rate",
].iloc[0]

highest_decile_default_rate = risk_decile_short.loc[
    risk_decile_short["decile"] == risk_decile_short["decile"].max(),
    "actual_default_rate",
].iloc[0]

highest_to_lowest_ratio = highest_decile_default_rate / lowest_decile_default_rate

decile_stats = pd.DataFrame({
    "lowest_decile_default_rate": [lowest_decile_default_rate],
    "highest_decile_default_rate": [highest_decile_default_rate],
    "highest_to_lowest_default_rate_ratio": [highest_to_lowest_ratio],
})

decile_stats

The risk decile table shows a strong and clean risk ranking pattern. Actual default rates increase from the lowest-risk decile to the highest-risk decile.

In the saved results, the lowest-risk decile has an actual default rate of about **4.27%**, while the highest-risk decile has an actual default rate of about **44.91%**. The highest-risk decile therefore defaults at about **10.5 times** the rate of the lowest-risk decile.

This is strong evidence that the baseline logistic regression model is useful for ranking loans by risk. Deciles 1 and 2 appear clearly safer, with actual default rates below 8%. Deciles 9 and 10 appear clearly risky, with actual default rates above 33% and 44%, respectively.

From a business perspective, this decile analysis is one of the most useful Milestone 2 outputs. It translates borrower-level probabilities into portfolio-level risk bands that are easier to explain and monitor.

### 5. Feature Interpretation

In [ ]:
# Extract logistic regression coefficients after preprocessing
preprocessor = pd_model.named_steps["preprocessor"]
logistic_model = pd_model.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()
coefficients = logistic_model.coef_[0]

coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
})

coefficient_table["absolute_coefficient"] = coefficient_table["coefficient"].abs()
coefficient_table = coefficient_table.sort_values("coefficient", ascending=False)

# Save the full coefficient table
coefficient_path = tables_dir / "pd_logistic_coefficients_v1.csv"
coefficient_table.to_csv(coefficient_path, index=False)

# Display the strongest positive and negative coefficients
n_top = 15
top_positive_coefficients = coefficient_table.head(n_top).copy()
top_negative_coefficients = coefficient_table.tail(n_top).sort_values("coefficient").copy()

print(f"Saved coefficient table to: {coefficient_path}")
display(top_positive_coefficients)
display(top_negative_coefficients)

Positive coefficients increase the model's predicted default risk. Negative coefficients decrease the model's predicted default risk.

In the current baseline model, the strongest risk-increasing coefficients include lower credit quality grades such as **grade G**, **grade F**, and **grade E**, along with some state and purpose categories. This is consistent with credit risk intuition because lower LendingClub grades generally represent higher borrower risk.

The strongest risk-decreasing coefficients include stronger credit grades such as **grade A**, **sub-grade A1**, **grade B**, and mortgage home ownership. This is also directionally intuitive because higher-quality grades should generally be associated with lower default risk.

These coefficients are directional model signals, not causal proof. A positive coefficient does not prove that a feature causes default, and a negative coefficient does not prove that a feature prevents default. Coefficients can also be affected by correlated variables, such as grade, sub-grade, and interest rate appearing together in the same model.

### 6. Credit Risk Interpretation

**Does the model separate high-risk and low-risk borrowers?**  
Yes. The risk decile results show that actual default rates rise from low-risk deciles to high-risk deciles. The highest-risk decile has an actual default rate more than ten times higher than the lowest-risk decile.

**Are the results consistent with credit risk intuition?**  
Yes. Higher-risk LendingClub grades and sub-grades increase predicted default risk, while stronger grades reduce predicted default risk. The decile pattern is also consistent with the idea that the model is ranking borrowers by credit risk.

**Are the predictions good enough for a first baseline?**  
Yes, for a first baseline. The ROC-AUC of 0.7094 and the clean decile separation suggest the model has learned meaningful risk patterns. However, this does not mean the model is ready for production or real credit decisions.

**What should be improved before relying on the model?**  
The model needs deeper leakage review, calibration review, feature refinement, stability testing, and comparison against other model types. The 0.5 classification threshold is especially not enough for lending decisions. Future work should focus on whether the predicted probabilities are stable, explainable, and reliable across time.

### 7. Limitations

The main limitations of this baseline model are:

- Logistic regression may not capture nonlinear patterns or feature interactions.
- One-hot encoding creates many variables, especially for categorical fields like state and sub-grade.
- Calibration may need improvement before PD values are used in Expected Loss calculations.
- The feature set is still basic and does not include deeper feature engineering.
- LendingClub data may not represent a bank's current credit portfolio or underwriting population.
- Some variables may overlap conceptually, especially `grade`, `sub_grade`, and `int_rate`.
- Coefficients are directional model signals, not causal explanations.
- The model has not yet been validated across time or loan issue year.
- Business cutoffs have not been selected or tested.

### 8. Outputs Created

In [ ]:
# Create standardized figure aliases for Milestone 2 reporting
# This does not change model training; it only saves clearer report filenames.
figure_aliases = {
    "pd_model_roc_curve_v1.png": "roc_curve_v1.png",
    "pd_model_precision_recall_curve_v1.png": "precision_recall_curve_v1.png",
    "pd_model_calibration_curve_v1.png": "calibration_curve_v1.png",
}

for source_name, target_name in figure_aliases.items():
    source_path = figures_dir / source_name
    target_path = figures_dir / target_name
    if source_path.exists():
        target_path.write_bytes(source_path.read_bytes())
        print(f"Saved standardized figure: {target_path}")
    else:
        print(f"Source figure not found, skipped: {source_path}")

Milestone 2 created the following output files:

```text
models/pd_logistic_regression_v1.pkl
reports/tables/pd_model_metrics_v1.csv
reports/tables/pd_risk_deciles_v1.csv
reports/tables/pd_logistic_coefficients_v1.csv
reports/figures/roc_curve_v1.png
reports/figures/precision_recall_curve_v1.png
reports/figures/calibration_curve_v1.png
reports/figures/default_rate_by_decile_v1.png
```

The original chart filenames from the modeling section are also retained in `reports/figures/`.

## What This Means for Milestone 3

Milestone 3 should focus on improving and validating the Probability of Default model.

Recommended next tasks:

- Compare logistic regression against Random Forest or XGBoost.
- Check whether `grade`, `sub_grade`, and `int_rate` create overlapping information.
- Improve calibration if needed.
- Review feature importance and coefficient stability.
- Test model stability across time or loan issue year.
- Create a cleaner final PD dataset with a more carefully reviewed feature set.
- Decide which PD model should feed Expected Loss later.

The baseline model is a solid first benchmark. The next milestone should test whether the model remains reliable when the feature set, model type, calibration approach, and validation method become more rigorous.